**Reading data from files**

In [2]:
from pathlib import Path

# Définir le chemin du répertoire
chemin = Path("../data/raw/first")

# Lister tout le contenu
content = ""
for element in chemin.iterdir():
    with open(element, 'r' , encoding='utf-8') as file:
        content = content  + "***" +file.read()


**start chunking**

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,          # Taille maximale du chunk (en caractères ou tokens)
    chunk_overlap=50,        # Recouvrement pour garder le contexte entre deux chunks
    separators="***"
)

c:\Users\EL OUARDI\Desktop\Spring bot\rag system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_core.documents import Document

chunks = [Document(page_content=part) for part in splitter.split_text(content)]

In [5]:
chunks

[Document(metadata={}, page_content='***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You Will Build\n\n\nYou will build an application that stores Customer objects in a memory-based database.\n\n\n***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You need\n\n\n\n\nAbout 15 minutes\n\nA favorite text editor or IDE\n\nJava 17 or later\n\nGradle 7.5+ or Maven 3.5+\n\nYou can also import the code straight into your IDE:\n\n\n\nSpring Tool Suite (STS)\n\nIntelliJ IDEA\n\nVSCode\n\n\n\n\n\n**'),
 Document(metadata={}, page_content='***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nHow to complete this guide\n\n\nLike most Spring Getting Started guides, you can start from scratch and complete each step or you can bypass basic setup steps that are already familiar to you. Either way, you end up with working code.\n\n\nTo start from sc

**Embedding**

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

# Remplacez votre OpenAIEmbeddings par ceci (qui va marche en local pour eviter les problemes de quota et de connexion au serveur d'openai ):
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1207.35it/s]


**create the victore store**

In [7]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    chunks, 
    embedding=embeddings, 
    collection_name="my_collection_spring_bot",
    persist_directory="./../vectore_store3"
    )

**Retrievling**

In [11]:
import yaml 

# Charger le fichier
with open("./../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

retriever = vectorstore.as_retriever(
    search_type="similarity",#pour le type de recherche que je veux faire, ici on utilise la recherche similarity qui est une recherche qui prend en compte la similarite entre les chunks et la question pour retourner les chunks les plus pertinents.
    search_kwargs={"k": config["retriever"]["top_k"]}#pour le nombre de chunks que je veux recuperer pour chaque question.
)

In [12]:
question = "how to use JPA to acces to a databse"
retrived_chunks = retriever.invoke(question)

In [13]:
retrived_chunks

[Document(metadata={}, page_content='***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nHow to complete this guide\n\n\nLike most Spring Getting Started guides, you can start from scratch and complete each step or you can bypass basic setup steps that are already familiar to you. Either way, you end up with working code.\n\n\nTo start from scratch, move on to Starting with Spring Initializr.\n\n\nTo skip the basics, do the following:\n\n\n\n\nDownload and unzip the source repository for this guide, or clone it using Git: git clone https://github.com/spring-guides/gs-accessing-data-jpa.git\n\ncd into gs-accessing-data-jpa/initial\n\nJump ahead to Define a Simple Entity.\n\n\n\nWhen you finish, you can check your results against the code in gs-accessing-data-jpa/complete.\n\n\n**'),
 Document(metadata={}, page_content='***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nHow to complete this g

**making prompt**

In [14]:
prompt_template = """
Tu es un assistant qui aide les utilisateurs a trouver des informations dans un document.
le contexte va etre entre <context> et </context> et la question d'utilisateur va etre entre <question> et </question>.
<context>
{context}
</context>
Voici la question de l'utilisateur :
<question>
{question}
</question>
Tu dois repondre a la question de l'utilisateur en utilisant le contexte que je t'ai fourni. Si tu ne trouves pas la reponse dans le contexte, dis que tu ne sais pas.
"""

In [15]:
relevant_docuemnt_chunks = retriever.invoke(question)
context_list = [chunk.page_content for chunk in relevant_docuemnt_chunks]
context = "\n".join(context_list)

In [16]:
prompt = prompt_template.format(context=context, question=question)

**Generation**

In [18]:
from openai import AsyncOpenAI  # Modification : Import asynchrone
from IPython.display import Markdown
import yaml 
from pathlib import Path


# Charger le fichier
with open("./../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)
with open("./../config/key.yaml", "r") as f:
    key_config = yaml.safe_load(f)

In [19]:
from openai import OpenAI
from IPython.display import Markdown

# Configuration du client pour OpenRouter
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=key_config["openrouterGeneratingModelKay"],
)

response = ""

# Envoi de la requête avec streaming
stream = client.chat.completions.create(
  model="deepseek/deepseek-v4-flash",
  messages=[
    {
      "role": "user",
      "content": prompt
    }
  ],
  stream=True,
  # Inclure les usage metadata pour recevoir les tokens de raisonnement à la fin
  stream_options={"include_usage": True} 
)


for chunk in stream:
    # Vérification du contenu (le texte généré)
    if chunk.choices and chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        response += content
        # print(content, end="", flush=True)


print(display(Markdown(response)))

D'après le contexte fourni, voici comment utiliser JPA pour accéder à une base de données avec Spring, en suivant le guide "Accessing Data with JPA" :

1. **Prérequis** : Vous avez besoin d'environ 15 minutes, d'un éditeur de texte ou IDE, de Java 17 ou version ultérieure, et de Gradle 7.5+ ou Maven 3.5+.

2. **Initialisation du projet** :
   - Rendez-vous sur [start.spring.io](https://start.spring.io)
   - Choisissez Gradle ou Maven et le langage souhaité
   - Saisissez "accessing-data-jpa" dans le champ "Artifact"
   - Ajoutez les dépendances **Spring Data JPA** et **H2 Database**
   - Générez et téléchargez le projet ZIP

3. **Construction de l'application** : Vous allez construire une application qui stocke des objets `Customer` dans une base de données en mémoire (H2).

4. **Étapes de réalisation** :
   - Pour partir de zéro, commencez avec Spring Initializr
   - Pour sauter les bases, téléchargez et décompressez le dépôt source, puis passez directement à l'étape "Define a Simple Entity"

5. **Vérification** : Une fois terminé, vous pouvez comparer votre code avec celui disponible dans le dossier `gs-accessing-data-jpa/complete`.

Le contexte ne fournit pas de détails plus précis sur la création de l'entité ou du repository, mais ces instructions vous permettent de démarrer un projet Spring Data JPA fonctionnel avec une base H2.

None


**Testing Distributed Systems**

In [21]:
import sys
import os

# 1. On identifie le dossier où se trouve votre projet
# Remplacez ce chemin par le chemin réel vers votre dossier "rag system"
project_root = r"C:\Users\EL OUARDI\Desktop\Spring bot\rag system"

# 2. On l'ajoute au chemin de recherche de Python
if project_root not in sys.path:
    sys.path.append(project_root)

In [22]:
from retriver.retriver import retreive_chunks
question  = "how to use jpa to acces to a database"
retreived_chunks = retreive_chunks(question)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1486.06it/s]
C:\Users\EL OUARDI\Desktop\Spring bot\rag system\retriver\vector_store_engin.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


C:\Users\EL OUARDI\Desktop\Spring bot\rag system\vectore_store3
C:\Users\EL OUARDI\Desktop\Spring bot\rag system\config\config.yaml
Nombre d'éléments dans la collection : 162
5
[Document(metadata={}, page_content='***\nSource Link: https://spring.io/guides/gs/spring-boot/\ntitle: Building an Application with Spring Boot\n\nHow to complete this guide\n\n\nLike most Spring Getting Started guides, you can start from scratch and complete each step or you can bypass basic setup steps that are already familiar to you. Either way, you end up with working code.\n\n\nTo start from scratch, move on to Starting with Spring Initializr.\n\n\nTo skip the basics, do the following:\n\n\n\n\nDownload and unzip the source repository for this guide, or clone it using Git: git clone https://github.com/spring-guides/gs-spring-boot.git\n\ncd into gs-spring-boot/initial\n\nJump ahead to Create a Simple Web Application.\n\n\n\nWhen you finish, you can check your results against the code in gs-spring-boot/comp

In [23]:
retreived_chunks

[Document(metadata={}, page_content='***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You Will Build\n\n\nYou will build an application that stores Customer objects in a memory-based database.\n\n\n***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You need\n\n\n\n\nAbout 15 minutes\n\nA favorite text editor or IDE\n\nJava 17 or later\n\nGradle 7.5+ or Maven 3.5+\n\nYou can also import the code straight into your IDE:\n\n\n\nSpring Tool Suite (STS)\n\nIntelliJ IDEA\n\nVSCode\n\n\n\n\n\n**'),
 Document(metadata={}, page_content='***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You Will Build\n\n\nYou will build an application that stores Customer objects in a memory-based database.\n\n\n***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You need\n\n\n\n\nAbout 15 minutes

In [24]:
from prompt.prompt_maker import make_prompt

prompt = make_prompt([doc.page_content for doc in retreived_chunks] , question)

In [25]:
prompt

"\nTu es un assistant qui aide les utilisateurs a trouver des informations dans un document.\nle contexte va etre entre <context> et </context> et la question d'utilisateur va etre entre <question> et </question>.\n<context>\n***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You Will Build\n\n\nYou will build an application that stores Customer objects in a memory-based database.\n\n\n***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You need\n\n\n\n\nAbout 15 minutes\n\nA favorite text editor or IDE\n\nJava 17 or later\n\nGradle 7.5+ or Maven 3.5+\n\nYou can also import the code straight into your IDE:\n\n\n\nSpring Tool Suite (STS)\n\nIntelliJ IDEA\n\nVSCode\n\n\n\n\n\n**\n***\nSource Link: https://spring.io/guides/gs/accessing-data-jpa\ntitle: Accessing Data with JPA\n\nWhat You Will Build\n\n\nYou will build an application that stores Customer objects in a memory-based data

In [27]:
from llm.llm import generate_response

response = await generate_response(prompt)

In [28]:
print(display(Markdown(response)))

D'après le contexte fourni, pour utiliser JPA afin d'accéder à une base de données, vous pouvez suivre le guide "Accessing Data with JPA" de Spring. Ce guide vous permet de :

- Créer une application qui stocke des objets `Customer` dans une base de données en mémoire.
- Utiliser Spring Data JPA pour sauvegarder et récupérer des objets depuis une base de données, sans avoir à écrire une implémentation concrète de repository.

Pour commencer, vous devez avoir :
- Java 17 ou version ultérieure
- Gradle 7.5+ ou Maven 3.5+
- Environ 15 minutes

Vous pouvez partir de zéro en utilisant Spring Initializr, ou bien télécharger/cloner le projet depuis GitHub : `git clone https://github.com/spring-guides/gs-accessing-data-jpa.git`.

Si vous souhaitez exposer les repositories JPA avec une interface REST hypermedia, le guide mentionne également la lecture de "Accessing JPA Data with REST".

None
